# Convergence-Based Optimization Analysis

Analysis of multi-start L-BFGS vs Bayesian optimization results from `opt_comp_convergence.py`.
Compares against brute force Sobol sampling baseline.

Avoids tau/tolerance-based performance profiles -- uses direct outcome metrics,
natural thresholds (brute force crossing), and integral measures (AUCC) instead.

In [ ]:
import os
import json
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import wilcoxon
from scipy.stats import t as student_t

sns.set_context('notebook')

# ------------------------------------------------------------------
# Load all convergence results (dynamic -- picks up any configs present)
# ------------------------------------------------------------------
CONV_DIR = '../examples/comparisons/closed_boundary_DIIID/convergence_02_16'
BF_DIR = '../examples/comparisons/closed_boundary_DIIID/brute_force'

rows = []
conv_curves = {}  # (coils, lam, method) -> np.array
skipped = []

for folder in sorted(os.listdir(CONV_DIR)):
    rpath = os.path.join(CONV_DIR, folder, 'results.json')
    if not os.path.isfile(rpath):
        continue
    m = re.match(r'lambda:([^,]+),coils:(\d+)', folder)
    if not m:
        continue
    lam = float(m.group(1))
    nc = int(m.group(2))

    try:
        with open(rpath) as f:
            data = json.load(f)
    except (json.JSONDecodeError, ValueError) as e:
        skipped.append(f"{folder}: {e}")
        continue

    bf_cost = data.get('brute_force_cost', None)

    for method_name, md in data['methods'].items():
        row = {
            'coils': nc,
            'lambda': lam,
            'method': method_name,
            'best_cost': md['best_cost'],
            'n_evals': md['n_evals'],
            'time': md['time'],
            'stopping': md['stopping'],
            'brute_force_cost': bf_cost,
        }
        if 'n_bayesian_evals' in md:
            row['n_bayesian_evals'] = md['n_bayesian_evals']
        if 'pts_refined' in md:
            row['pts_refined'] = md['pts_refined']
        rows.append(row)

        conv_curves[(nc, lam, method_name)] = np.array(md['convergence_history'])

df = pd.DataFrame(rows)

# Load brute force results
bf_rows = []
for folder in sorted(os.listdir(BF_DIR)):
    rpath = os.path.join(BF_DIR, folder, 'results.json')
    if not os.path.isfile(rpath):
        continue
    m = re.match(r'lambda:([^,]+),coils:(\d+)', folder)
    if not m:
        continue
    try:
        with open(rpath) as f:
            bfd = json.load(f)
    except (json.JSONDecodeError, ValueError):
        continue
    bf_rows.append({
        'coils': int(m.group(2)),
        'lambda': float(m.group(1)),
        'bf_best_cost': bfd['best_cost'],
        'bf_n_evals': bfd['n_evals'],
        'bf_time': bfd['time'],
    })
bf_df = pd.DataFrame(bf_rows)

configs = df.groupby(['coils', 'lambda']).size().reset_index(name='n_methods')
print(f"Loaded {len(configs)} configurations, {len(df)} total method results")
print(f"Coils: {sorted(df['coils'].unique())}")
print(f"Lambda: {sorted(df['lambda'].unique())}")
print(f"Methods: {sorted(df['method'].unique())}")
print(f"Brute force configs: {len(bf_df)}")
if skipped:
    print(f"Skipped {len(skipped)} corrupted files:")
    for s in skipped:
        print(f"  {s}")

In [ ]:
# ------------------------------------------------------------------
# Summary table: per-config comparison
# ------------------------------------------------------------------
lbfgs = df[df['method'] == 'Multi-start L-BFGS'].set_index(['coils', 'lambda'])
bayes = df[df['method'] == 'Bayesian'].set_index(['coils', 'lambda'])

common_idx = lbfgs.index.intersection(bayes.index)

summary_rows = []
for idx in sorted(common_idx):
    nc, lam = idx
    lc = lbfgs.loc[idx, 'best_cost']
    bc = bayes.loc[idx, 'best_cost']
    bf = lbfgs.loc[idx, 'brute_force_cost']

    best_of_two = min(lc, bc)
    gap_pct = abs(lc - bc) / best_of_two * 100 if best_of_two > 0 else 0

    if gap_pct < 0.1:
        winner = 'Tie'
    elif lc < bc:
        winner = 'L-BFGS'
    else:
        winner = 'Bayesian'

    summary_rows.append({
        'coils': nc,
        'lambda': lam,
        'brute_force': bf,
        'lbfgs_cost': lc,
        'bayesian_cost': bc,
        'winner': winner,
        'cost_gap_pct': gap_pct,
        'lbfgs_evals': int(lbfgs.loc[idx, 'n_evals']),
        'bayesian_evals': int(bayes.loc[idx, 'n_evals']),
        'lbfgs_time': lbfgs.loc[idx, 'time'],
        'bayesian_time': bayes.loc[idx, 'time'],
    })

summary = pd.DataFrame(summary_rows)
display(summary.style.format({
    'brute_force': '{:.4e}',
    'lbfgs_cost': '{:.4e}',
    'bayesian_cost': '{:.4e}',
    'cost_gap_pct': '{:.2f}%',
    'lbfgs_time': '{:.1f}s',
    'bayesian_time': '{:.1f}s',
}))

In [ ]:
# ------------------------------------------------------------------
# Cost improvement over brute force (heatmaps)
# ------------------------------------------------------------------
summary['lbfgs_improv_pct'] = (summary['brute_force'] - summary['lbfgs_cost']) / summary['brute_force'] * 100
summary['bayes_improv_pct'] = (summary['brute_force'] - summary['bayesian_cost']) / summary['brute_force'] * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [(ax1, 'lbfgs_improv_pct', 'L-BFGS improvement over brute force (%)'),
                        (ax2, 'bayes_improv_pct', 'Bayesian improvement over brute force (%)')]:
    pivot = summary.pivot_table(index='coils', columns='lambda', values=col)
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGn', ax=ax,
                cbar_kws={'label': '% improvement'})
    ax.set_title(title)
    ax.set_xlabel('lambda')
    ax.set_ylabel('coils')

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Evals to beat brute force
# ------------------------------------------------------------------
crossing_rows = []
for _, row in summary.iterrows():
    nc, lam = row['coils'], row['lambda']
    bf = row['brute_force']
    if bf is None or np.isnan(bf):
        continue

    cr = {'coils': nc, 'lambda': lam, 'brute_force': bf}
    for method in ['Multi-start L-BFGS', 'Bayesian']:
        key = (nc, lam, method)
        if key not in conv_curves:
            continue
        curve = conv_curves[key]
        below = np.where(curve < bf)[0]
        crossing = int(below[0]) + 1 if len(below) > 0 else np.nan
        short = 'lbfgs' if 'L-BFGS' in method else 'bayesian'
        cr[f'{short}_crossing'] = crossing

    crossing_rows.append(cr)

crossing_df = pd.DataFrame(crossing_rows)
crossing_df['ratio'] = crossing_df['bayesian_crossing'] / crossing_df['lbfgs_crossing']

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title, cmap in [
    (ax1, 'lbfgs_crossing', 'L-BFGS evals to beat brute force', 'YlOrRd_r'),
    (ax2, 'bayesian_crossing', 'Bayesian evals to beat brute force', 'YlOrRd_r'),
    (ax3, 'ratio', 'Ratio (Bayesian / L-BFGS)', 'RdYlGn_r'),
]:
    pivot = crossing_df.pivot_table(index='coils', columns='lambda', values=col)
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    fmt = '.0f' if col != 'ratio' else '.2f'
    sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('lambda')
    ax.set_ylabel('coils')

plt.tight_layout()
plt.show()

display(crossing_df.style.format({
    'brute_force': '{:.4e}',
    'lbfgs_crossing': '{:.0f}',
    'bayesian_crossing': '{:.0f}',
    'ratio': '{:.2f}',
}))

In [ ]:
# ------------------------------------------------------------------
# Sample efficiency: n_evals ratio
# ------------------------------------------------------------------
summary['eval_ratio'] = summary['bayesian_evals'] / summary['lbfgs_evals']

fig, ax = plt.subplots(figsize=(7, 5))
pivot = summary.pivot_table(index='coils', columns='lambda', values='eval_ratio')
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=ax,
            center=1.0, cbar_kws={'label': 'Bayesian / L-BFGS evals'})
ax.set_title('Sample efficiency ratio (< 1 = Bayesian uses fewer evals)')
ax.set_xlabel('lambda')
ax.set_ylabel('coils')
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Wall-clock time by coils and lambda
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title in [
    (axes[0], 'lbfgs_time',    'L-BFGS wall time (s)'),
    (axes[1], 'bayesian_time', 'Bayesian wall time (s)'),
]:
    pivot = summary.pivot_table(index='coils', columns='lambda', values=col)
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
                cbar_kws={'label': 'seconds'})
    ax.set_title(title)
    ax.set_xlabel('lambda')
    ax.set_ylabel('coils')

summary['time_ratio'] = summary['bayesian_time'] / summary['lbfgs_time']
pivot_ratio = summary.pivot_table(index='coils', columns='lambda', values='time_ratio')
pivot_ratio = pivot_ratio.reindex(sorted(pivot_ratio.columns), axis=1)
sns.heatmap(pivot_ratio, annot=True, fmt='.2f', cmap='RdYlGn_r',
            ax=axes[2], center=1.0,
            cbar_kws={'label': 'Bayesian / L-BFGS'})
axes[2].set_title('Time ratio: Bayesian / L-BFGS\n(> 1 = Bayesian slower)')
axes[2].set_xlabel('lambda')
axes[2].set_ylabel('coils')

plt.tight_layout()
plt.show()

display(summary[['coils', 'lambda', 'lbfgs_time', 'bayesian_time', 'time_ratio']].sort_values(
    ['coils', 'lambda']).style.format({
        'lbfgs_time': '{:.1f}s',
        'bayesian_time': '{:.1f}s',
        'time_ratio': '{:.2f}',
    }))

In [ ]:
# ------------------------------------------------------------------
# Statistical tests
# ------------------------------------------------------------------
rng = np.random.default_rng(42)

def geometric_mean_ci(values, reps=10000, alpha=0.05):
    logs = np.log(values)
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan
    gmean = float(np.exp(logs.mean()))
    idx = rng.integers(0, n, size=(reps, n))
    boot_means = logs[idx].mean(axis=1)
    lo, hi = np.percentile(boot_means, [100*alpha/2, 100*(1-alpha/2)])
    return gmean, float(np.exp(lo)), float(np.exp(hi))

# Build brute force crossing ratio array aligned with summary
bf_crossing_ratio = np.full(len(summary), np.nan)
for i, (_, row) in enumerate(summary.iterrows()):
    nc, lam = row['coils'], row['lambda']
    match = crossing_df[(crossing_df['coils'] == nc) & (crossing_df['lambda'] == lam)]
    if not match.empty:
        bf_crossing_ratio[i] = match.iloc[0]['lbfgs_crossing'] / match.iloc[0]['bayesian_crossing']

test_rows = []
for metric_name, vals_array in [
    ('Cost (L-BFGS/Bayesian)',
     summary['lbfgs_cost'].values / summary['bayesian_cost'].values),
    ('Total evals (L-BFGS/Bayesian)',
     summary['lbfgs_evals'].values / summary['bayesian_evals'].values),
    ('Evals to beat brute force (L-BFGS/Bayesian)',
     bf_crossing_ratio),
]:
    vals = vals_array[np.isfinite(vals_array) & (vals_array > 0)]
    log_vals = np.log(vals)

    gm, gm_lo, gm_hi = geometric_mean_ci(vals)

    wp = np.nan
    nonzero = log_vals[log_vals != 0]
    if len(nonzero) > 0:
        try:
            res = wilcoxon(nonzero, alternative='two-sided')
            wp = float(res.pvalue)
        except Exception:
            pass

    test_rows.append({
        'metric': metric_name,
        'n': len(vals),
        'geom_mean': gm,
        'ci_low': gm_lo,
        'ci_high': gm_hi,
        'wilcoxon_p': wp,
    })

test_df = pd.DataFrame(test_rows)
display(test_df.style.format({
    'geom_mean': '{:.4f}',
    'ci_low': '{:.4f}',
    'ci_high': '{:.4f}',
    'wilcoxon_p': '{:.4g}',
}))

print("\nInterpretation:")
print("  Ratio > 1 means Bayesian better (L-BFGS used more evals / found higher cost)")
print("  geom_mean > 1 with wilcoxon_p < 0.05: Bayesian systematically better")

In [ ]:
# ------------------------------------------------------------------
# Monotonic trend tests: Spearman + Kendall (nonparametric)
# ------------------------------------------------------------------
# Tests whether ratio systematically increases/decreases with coils or lambda,
# with NO assumption about functional form (linear, quadratic, etc.).
#
# Spearman rho: correlation between ranks
# Kendall tau: fraction of concordant vs discordant pairs (more robust at small n)
#
# These directly answer: "as coils increase (or lambda decreases),
# does the ratio significantly tend to go up?"

from scipy.stats import spearmanr, kendalltau

trend_rows = []
for ratio_name, ratio_vals, coils_src, lam_src in [
    ('Cost (L-BFGS/Bayesian)',
     summary['lbfgs_cost'].values / summary['bayesian_cost'].values,
     summary['coils'].values, summary['lambda'].values),
    ('Total evals (L-BFGS/Bayesian)',
     summary['lbfgs_evals'].values / summary['bayesian_evals'].values,
     summary['coils'].values, summary['lambda'].values),
    ('Evals to beat brute force (L-BFGS/Bayesian)',
     bf_crossing_ratio,
     summary['coils'].values, summary['lambda'].values),
]:
    valid = np.isfinite(ratio_vals) & (ratio_vals > 0)
    log_ratio = np.log(ratio_vals[valid])
    coils_arr = coils_src[valid].astype(float)
    log_lam = np.log(lam_src[valid].astype(float))

    for predictor_name, predictor in [('coils', coils_arr), ('log(lambda)', log_lam)]:
        sp_rho, sp_p = spearmanr(predictor, log_ratio)
        kt_tau, kt_p = kendalltau(predictor, log_ratio)

        trend_rows.append({
            'response': ratio_name,
            'predictor': predictor_name,
            'spearman_rho': float(sp_rho),
            'spearman_p': float(sp_p),
            'kendall_tau': float(kt_tau),
            'kendall_p': float(kt_p),
            'n': int(valid.sum()),
        })

trend_df = pd.DataFrame(trend_rows)
display(trend_df.style.format({
    'spearman_rho': '{:+.4f}',
    'spearman_p': '{:.4g}',
    'kendall_tau': '{:+.4f}',
    'kendall_p': '{:.4g}',
}))

print("\nInterpretation (ratio = L-BFGS / Bayesian, so ratio > 1 = Bayesian better):")
print("  coils with positive rho/tau: more coils -> Bayesian advantage grows")
print("  log(lambda) with negative rho/tau: smaller lambda -> Bayesian advantage grows")
print("  p < 0.05: monotonic trend is statistically significant")

In [ ]:
# ------------------------------------------------------------------
# Direct cost comparison: L-BFGS vs Bayesian
# ------------------------------------------------------------------
summary['cost_ratio'] = summary['lbfgs_cost'] / summary['bayesian_cost']
summary['cost_gap_pct'] = (summary['lbfgs_cost'] - summary['bayesian_cost']) / summary[['lbfgs_cost', 'bayesian_cost']].min(axis=1) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

pivot_ratio = summary.pivot_table(index='coils', columns='lambda', values='cost_ratio')
pivot_ratio = pivot_ratio.reindex(sorted(pivot_ratio.columns), axis=1)
sns.heatmap(
    pivot_ratio, annot=True, fmt='.3f', cmap='RdYlGn',
    ax=ax1, center=1.0,
    cbar_kws={'label': 'L-BFGS cost / Bayesian cost'}
)
ax1.set_title('Cost ratio: L-BFGS / Bayesian\n(> 1 = Bayesian found better minimum)')
ax1.set_xlabel('lambda')
ax1.set_ylabel('coils')

pivot_gap = summary.pivot_table(index='coils', columns='lambda', values='cost_gap_pct')
pivot_gap = pivot_gap.reindex(sorted(pivot_gap.columns), axis=1)
sns.heatmap(
    pivot_gap, annot=True, fmt='.1f', cmap='RdYlGn_r',
    ax=ax2, center=0,
    cbar_kws={'label': '%'}
)
ax2.set_title('Cost gap: (L-BFGS - Bayesian) / min * 100\n(positive = L-BFGS worse)')
ax2.set_xlabel('lambda')
ax2.set_ylabel('coils')

plt.tight_layout()
plt.show()